In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  


import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df_noodle = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/B물류센터 매출_2021~2024_봉지라면_전치리.csv')
df_noodle.head()
df_noodle['판매일'] = pd.to_datetime(df_noodle['판매일'])

df_carbon = pd.read_csv('./수요예측부문 데이터 제공/csv파일 모음/B물류센터 매출_2021~2024_탄산음료_전치리.csv')
df_carbon.head()
df_carbon['판매일'] = pd.to_datetime(df_carbon['판매일'])

df_noodle['연월'] = df_noodle['판매일'].dt.to_period('M')
df_carbon['연월'] = df_carbon['판매일'].dt.to_period('M')

df_noodle_time = df_noodle.groupby('연월')['판매수량'].sum().reset_index()
df_carbon_time = df_carbon.groupby('연월')['판매수량'].sum().reset_index()

df_noodle_time['연월'] = df_noodle_time['연월'].dt.to_timestamp()
df_carbon_time['연월'] = df_carbon_time['연월'].dt.to_timestamp()

In [ ]:
df_noodle_time.to_csv('./수요예측부문 데이터 제공/csv파일 모음/봉지라면월별시계열.csv')
df_carbon_time.to_csv('./수요예측부문 데이터 제공/csv파일 모음/탄산음료월별시계열.csv')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# ===== 데이터 준비 =====
df = df_noodle_time.copy()
df.set_index('연월', inplace=True)
df = df.asfreq('MS')  # 월별 데이터로 설정
# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5          # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []    # fold별 예측값 저장용

# ===== Rolling CV =====
for i in range(1, folds+1):
    split_point = len(df) - (folds - i)*h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # SARIMA(1,1,1)(0,1,1,12) 모델
    model = SARIMAX(train['판매수량'],
                    order=(1,1,1),
                    seasonal_order=(0,1,1,12),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====

plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

# 각 fold별 예측 구간 시각화
for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('SARIMA(1,1,1)(1,1,1,12) Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# ===== 데이터 준비 =====
df = df_noodle_time.copy()
df.set_index('연월', inplace=True)

# 인덱스를 datetime으로 변환 (Period, str 대비)
df.index = pd.to_datetime(df.index, errors='coerce')

# 월별 데이터로 설정
df = df.asfreq('MS')  
df['판매수량'] = df['판매수량'].fillna(method='ffill')

# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5            # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []    # fold별 예측값 저장용

# ===== Rolling CV =====
for i in range(1, folds+1):
    split_point = len(df) - (folds - i)*h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # ✅ ARIMA(0,1,1) 모델 (비계절형)
    model = SARIMAX(train['판매수량'],
                    order=(1,1,1),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====
plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

# 각 fold별 예측 구간 시각화
for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('ARIMA(0,1,1) Rolling Forecast (3개월 단위, 5-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 센터B 탄산음료

In [ ]:
# 1행 2열로 시각화
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plot_acf(df_carbon_time['판매수량'], lags=24, ax=plt.gca())
plt.title("ACF")
plt.subplot(1,2,2)
plot_pacf(df_carbon_time['판매수량'], lags=24, ax=plt.gca())
plt.title("PACF")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# ===== 데이터 준비 =====
df = df_carbon_time.copy()
df.set_index('연월', inplace=True)
df = df.asfreq('MS')  # 월별 데이터로 설정
# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5          # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

rmse_list = []
pred_df_list = []    # fold별 예측값 저장용

# ===== Rolling CV =====
for i in range(1, folds+1):
    split_point = len(df) - (folds - i)*h
    train = df.iloc[:split_point - h]
    test = df.iloc[split_point - h:split_point]

    if len(train) < season_length:
        continue

    # SARIMA(1,1,1)(0,1,1,12) 모델
    model = SARIMAX(train['판매수량'],
                    order=(0,1,1),
                    seasonal_order=(0,1,1,12),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    result = model.fit(disp=False)

    # 3개월 ahead 예측
    forecast = result.forecast(steps=h)
    forecast = pd.Series(forecast, index=test.index, name=f'fold{i}_예측')

    rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
    rmse_list.append(rmse)
    pred_df_list.append(pd.DataFrame({'실제': test['판매수량'], '예측': forecast}))

    print(f"Fold {i}: Train={train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}, "
          f"Test={test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}, RMSE={rmse:,.0f}")

# ===== 결과 요약 =====
print("\n📊 평균 RMSE:", round(np.mean(rmse_list), 2))

# ===== 시각화 =====

plt.figure(figsize=(12,6))
plt.plot(df.index, df['판매수량'], label='Actual', color='black')

# 각 fold별 예측 구간 시각화
for i, pred in enumerate(pred_df_list):
    plt.plot(pred.index, pred['예측'], label=f'Fold {i+1} Forecast', linestyle='--')

plt.title('SARIMA(1,1,1)(0,1,1,12) Rolling Forecast (3개월 단위, 10-Fold)')
plt.xlabel('날짜')
plt.ylabel('판매수량')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 25년 1, 2, 3예측

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

import warnings
warnings.filterwarnings('ignore')

# ===== 데이터 준비 =====

df = df.asfreq('MS')

# ===== 파라미터 설정 =====
season_length = 12   # 월별 데이터니까 12개월 주기
folds = 5           # 교차검증 fold 개수
h = 3                # 한 번에 예측할 기간 (3개월 ahead)

import itertools

# 후보 파라미터 조합
param_list = [
    ((1,1,1), (1,1,1,12)),
    ((0,1,1), (0,1,1,12)),
    ((1,1,1), (0,1,1,12)),
    ((2,1,1), (1,1,0,12)),
    ((1,1,2), (0,1,1,12)),
    ((2,1,2), (1,1,1,12)),

]

results = []

for (order, seasonal_order) in param_list:
    rmse_list = []
    print(f"\n===== 모델: SARIMA{order}x{seasonal_order} =====")

    for i in range(1, folds+1):
        split_point = len(df) - (folds - i)*h
        train = df.iloc[:split_point - h]
        test = df.iloc[split_point - h:split_point]

        if len(train) < season_length:
            continue

        model = SARIMAX(train['판매수량'],
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        result = model.fit(disp=False)

        forecast = result.forecast(steps=h)
        rmse = np.sqrt(mean_squared_error(test['판매수량'], forecast))
        rmse_list.append(rmse)

    avg_rmse = np.mean(rmse_list)
    results.append((order, seasonal_order, avg_rmse))
    print(f"📊 평균 RMSE = {avg_rmse:,.2f}")

# 결과를 표로 정리
results_df = pd.DataFrame(results, columns=['(p,d,q)', '(P,D,Q,s)', '평균_RMSE']).sort_values('평균_RMSE')
print("\n===== 모델별 평균 RMSE 비교 =====")
print(results_df)


In [ ]:
df_carbon_time = df_carbon_time.copy()

model = SARIMAX(df_carbon_time['판매수량'],
                order=(0,1,1),
                seasonal_order=(0,1,1,12),
                enforce_stationarity=False,
                enforce_invertibility=False)

# 3개월 예측

result = model.fit(disp=False)
forecast = result.get_forecast(steps=3)

pred_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

# 5️⃣ 결과 출력
print("📅 예측 결과:")
print(pred_mean)
print("\n📈 신뢰구간:")
print(conf_int)

# 6️⃣ 시각화
plt.figure(figsize=(10,4))
plt.plot(df_carbon_time.index, df_carbon_time['판매수량'], label='Actual')
plt.plot(pred_mean.index, pred_mean, label='Forecast', color='red', marker='o')
plt.fill_between(conf_int.index,
                 conf_int.iloc[:,0],
                 conf_int.iloc[:,1],
                 color='pink', alpha=0.3)
plt.title('SARIMA 3-Month Forecast (탄산음료)')
plt.legend()
plt.show()


In [ ]:
# 생수, 음료, 건강 
# ===== 모델: ARIMA(0,1,1) =====

df_noodle_time = df_noodle_time.copy()

# ✅ ARIMA(0,1,1) 모델 (비계절형)
model = SARIMAX(df_noodle_time['판매수량'],
                    order=(0,1,1),
                    enforce_stationarity=False,
                    enforce_invertibility=False)

# 3개월 예측

result = model.fit(disp=False)
forecast = result.get_forecast(steps=3)

pred_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

# 5️⃣ 결과 출력
print("📅 예측 결과:")
print(pred_mean)
print("\n📈 신뢰구간:")
print(conf_int)

# 6️⃣ 시각화
plt.figure(figsize=(10,4))
plt.plot(df_noodle_time.index, df_noodle_time['판매수량'], label='Actual')
plt.plot(pred_mean.index, pred_mean, label='Forecast', color='red', marker='o')
plt.fill_between(conf_int.index,
                 conf_int.iloc[:,0],
                 conf_int.iloc[:,1],
                 color='pink', alpha=0.3)
plt.title('SARIMA 3-Month Forecast (생수/음료/건강)')
plt.legend()
plt.show()

